# Fine-tune BERT on Sentiment140 (Kaggle)

This notebook downloads Stanford's raw Sentiment140 ZIP and parses its CSV locally in Python. It does not use `datasets.load_dataset` or execute dataset repository code. It cleans the tweets, maps labels to negative/positive, creates stratified 80/10/10 train/validation/test splits, fine-tunes BERT, and saves metrics and model artifacts under `/kaggle/working/sentiment140-bert`.

## Before running
In Kaggle Notebook Settings, enable **Internet** for the Stanford archive and model-weight downloads, and choose a **GPU accelerator**. Hosted GPU training moves the compute off your laptop, so it avoids heating your laptop from local model training. GPU availability and weekly quotas depend on your Kaggle account. Sentiment140 labels are noisy; the training labels are assigned using emoticons.

In [8]:
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import csv
import html
import io
import json
import random
import re
import shutil
import unicodedata
import zipfile
from pathlib import Path
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

SEED = 42
SAMPLE_SIZE = 20_000  # Set to None to use all cleaned training tweets.
MODEL_NAME = 'bert-base-uncased'
MAX_LENGTH = 128
EPOCHS = 2
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
DATA_DIR = Path('/kaggle/working/sentiment140-data')
OUTPUT_DIR = Path('/kaggle/working/sentiment140-bert')
ARCHIVE_PATH = DATA_DIR / 'raw' / 'sentiment140.zip'
ARCHIVE_URL = 'https://cs.stanford.edu/people/alecmgo/trainingandtestdata.zip'
TRAIN_CSV = 'training.1600000.processed.noemoticon.csv'

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## Download and parse the raw Stanford archive

The original training CSV has six fields: polarity, tweet ID, date, query, user, and tweet text. This cell reads only the label and text using Python's CSV parser. The downloaded ZIP is cached in the Kaggle working directory.

In [9]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH.parent.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    request = Request(ARCHIVE_URL, headers={'User-Agent': 'sentiment-analysis-kaggle/1.0'})
    temporary_path = ARCHIVE_PATH.with_suffix('.zip.part')
    print('Downloading Sentiment140 from Stanford...')
    with urlopen(request, timeout=60) as response, temporary_path.open('wb') as output_file:
        shutil.copyfileobj(response, output_file)
    temporary_path.replace(ARCHIVE_PATH)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    member = next((name for name in archive.namelist() if Path(name).name == TRAIN_CSV), None)
    if member is None:
        raise FileNotFoundError(f'{TRAIN_CSV} not found in {ARCHIVE_PATH}')
    rows = []
    with archive.open(member) as binary_file:
        text_file = io.TextIOWrapper(binary_file, encoding='ISO-8859-1', newline='')
        for row_number, row in enumerate(csv.reader(text_file, delimiter=',', quotechar='"'), start=1):
            if len(row) != 6:
                raise ValueError(f'Malformed source row {row_number}: expected 6 fields, found {len(row)}')
            rows.append((row[5], int(row[0])))

raw_df = pd.DataFrame(rows, columns=['text', 'sentiment'])
print(f'Loaded {len(raw_df):,} tweets from {TRAIN_CSV}')
raw_df.head()

Loaded 1,600,000 tweets from training.1600000.processed.noemoticon.csv


,text,sentiment
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",0
1,is upset that he can't update his Facebook by ...,0
2,@Kenichan I dived many times for the ball. Man...,0
3,my whole body feels itchy and like its on fire,0
4,"@nationwideclass no, it's not behaving at all....",0


## Clean, encode, deduplicate, and split

URLs and user mentions become consistent tokens, hashtag markers are removed while retaining their words, and punctuation/emojis are kept. Label `0` maps to negative and `4` to positive. The random held-out test partition is never used for epoch selection.

In [10]:
URL_RE = re.compile(r'https?://\S+|www\.\S+', re.IGNORECASE)
MENTION_RE = re.compile(r'(?<!\w)@[A-Za-z0-9_]+')
HTML_RE = re.compile(r'<[^>]+>')
SPACE_RE = re.compile(r'\s+')

def clean_text(value):
    if value is None or pd.isna(value):
        return ''
    text = html.unescape(unicodedata.normalize('NFKC', str(value)))
    text = HTML_RE.sub(' ', text)
    text = URL_RE.sub(' URL ', text)
    text = MENTION_RE.sub(' USER ', text)
    text = ''.join(ch for ch in text if ch.isprintable() or ch in '\n\t')
    return SPACE_RE.sub(' ', text.replace('#', '')).strip()

df = raw_df.copy()
df['text'] = df['text'].map(clean_text)
df = df[df['text'].str.len() > 0].copy()
df['label'] = df['sentiment'].map({0: 0, 4: 1})
if df['label'].isna().any():
    raise ValueError('Unexpected source sentiment labels; expected only 0 and 4.')
df = df[['text', 'label']].drop_duplicates(subset='text').reset_index(drop=True)
if SAMPLE_SIZE and len(df) > SAMPLE_SIZE:
    df, _ = train_test_split(df, train_size=SAMPLE_SIZE, stratify=df['label'], random_state=SEED)
    df = df.reset_index(drop=True)

train_validation_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df['label'], random_state=SEED
)
train_df, validation_df = train_test_split(
    train_validation_df, test_size=1/9, stratify=train_validation_df['label'], random_state=SEED
)
splits = {
    'train': train_df.reset_index(drop=True),
    'validation': validation_df.reset_index(drop=True),
    'test': test_df.reset_index(drop=True),
}
processed_dir = DATA_DIR / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)
for name, split_df in splits.items():
    split_df.to_csv(processed_dir / f'{name}.csv', index=False)
    print(f'{name:>10}: {len(split_df):,} rows; labels {split_df.label.value_counts().sort_index().to_dict()}')

     train: 16,000 rows; labels {0: 8049, 1: 7951}
validation: 2,000 rows; labels {0: 1006, 1: 994}
      test: 2,000 rows; labels {0: 1006, 1: 994}


## Tokenize and fine-tune BERT

The validation set is evaluated each epoch and selects the best checkpoint by F1. Training uses mixed precision when CUDA is available to reduce GPU memory use.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2,
    id2label={0: 'negative', 1: 'positive'},
    label2id={'negative': 0, 'positive': 1},
)

class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.labels = frame['label'].astype(int).tolist()
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, index):
        item = tokenizer(self.texts[index], truncation=True, max_length=MAX_LENGTH)
        item['labels'] = self.labels[index]
        return item

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0),
    }

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    report_to='none',
    seed=SEED,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    save_total_limit=2,
)
trainer = Trainer(
    model=model, args=training_args,
    train_dataset=TweetDataset(splits['train']),
    eval_dataset=TweetDataset(splits['validation']),
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
train_result = trainer.train()
print('Training complete:', train_result.metrics)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.402700,0.419888,0.815000,0.806483,0.825956,0.816103
2,0.290500,0.438774,0.823000,0.820000,0.824950,0.822467


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training complete: {'train_runtime': 224.7263, 'train_samples_per_second': 142.395, 'train_steps_per_second': 4.45, 'total_flos': 680487411365760.0, 'train_loss': 0.3784327983856201, 'epoch': 2.0}


## Final validation and held-out test evaluation

The test split is evaluated only after training and checkpoint selection are complete. The model, tokenizer, split CSV files, metrics, and training history are written to Kaggle's output directory so they can be downloaded from the notebook's Output panel.

In [12]:
validation_metrics = trainer.evaluate(
    eval_dataset=TweetDataset(splits['validation']), metric_key_prefix='validation'
)
test_output = trainer.predict(
    TweetDataset(splits['test']), metric_key_prefix='test'
)
test_metrics = test_output.metrics
print('Validation metrics:', validation_metrics)
print('Held-out test metrics:', test_metrics)

trainer.save_model(str(OUTPUT_DIR / 'model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'model'))
with (OUTPUT_DIR / 'validation_metrics.json').open('w') as f:
    json.dump(validation_metrics, f, indent=2)
with (OUTPUT_DIR / 'test_metrics.json').open('w') as f:
    json.dump(test_metrics, f, indent=2)
with (OUTPUT_DIR / 'training_history.json').open('w') as f:
    json.dump(trainer.state.log_history, f, indent=2)
print('Saved outputs to:', OUTPUT_DIR)
sorted(path.name for path in OUTPUT_DIR.iterdir())

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation metrics: {'validation_loss': 0.4387744069099426, 'validation_accuracy': 0.823, 'validation_precision': 0.82, 'validation_recall': 0.8249496981891348, 'validation_f1': 0.8224674022066198, 'validation_runtime': 5.0005, 'validation_samples_per_second': 399.96, 'validation_steps_per_second': 12.599, 'epoch': 2.0}
Held-out test metrics: {'test_loss': 0.41885918378829956, 'test_accuracy': 0.8255, 'test_precision': 0.8183613030602171, 'test_recall': 0.8340040241448692, 'test_f1': 0.8261086198305929, 'test_runtime': 4.9516, 'test_samples_per_second': 403.906, 'test_steps_per_second': 12.723}
Saved outputs to: /kaggle/working/sentiment140-bert


['checkpoints',
 'model',
 'test_metrics.json',
 'training_history.json',
 'validation_metrics.json']